In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
from pathlib import Path

DATA_DIR = Path("../data/intermediate")
OUT_DIR = Path("../out")
PEGEL_PATH = DATA_DIR / "pegel_kirchentellinsfurt.csv"
WEATHER_PATH = DATA_DIR / "weather_data_daily_2000_2026.csv"
WEATHER_HOURLY_PATH = DATA_DIR / "weather_data_hourly_2000_2026.csv"
MAX_LAG_DAYS = 3


def load_data(pegel_path=PEGEL_PATH, weather_path=WEATHER_PATH):
    pegel = pd.read_csv(pegel_path, sep=";", usecols=["Messzeitpunkt", "Messwert"], dtype="string")
    pegel["timestamp"] = pd.to_datetime(pegel.pop("Messzeitpunkt"), format="%d.%m.%Y %H:%M", errors="coerce")
    pegel["water_level"] = pd.to_numeric(pegel.pop("Messwert").str.replace(",", ".", regex=False), errors="coerce")
    pegel = pegel.dropna(subset=["timestamp", "water_level"]).sort_values("timestamp")
    pegel["date"] = pegel["timestamp"].dt.normalize()

    weather = pd.read_csv(weather_path, parse_dates=["timestamp"])
    weather["timestamp"] = weather["timestamp"].dt.normalize()
    weather["station_id"] = weather["station_id"].astype(str).str.zfill(5)
    return pegel, weather


def load_hourly_weather(weather_path=WEATHER_HOURLY_PATH):
    weather = pd.read_csv(weather_path, parse_dates=["timestamp"])
    weather["timestamp"] = weather["timestamp"].dt.floor("h")
    weather["station_id"] = weather["station_id"].astype(str).str.zfill(5)
    return weather


def correlation_by_lag(pegel, weather_station, max_lag_days=MAX_LAG_DAYS):
    """Positive lag d compares water at t with station weather from t-d days."""
    joined = pegel.merge(weather_station, left_on="date", right_on="timestamp", how="left")
    weather_columns = [column for column in weather_station.columns if column not in {"station_id", "timestamp"}]
    results = []
    water = joined["water_level"]
    for lag_days in range(max_lag_days + 1):
        lag_hours = 24 * lag_days
        shifted_water = water.shift(lag_hours)
        results.append({"variable": "water_level", "lag_days": lag_days,
                        "correlation": water.corr(shifted_water),
                        "n_pairs": int((water.notna() & shifted_water.notna()).sum())})
        for column in weather_columns:
            shifted_weather = joined[column].shift(lag_hours)
            valid = water.notna() & shifted_weather.notna()
            results.append({"variable": column, "lag_days": lag_days,
                            "correlation": water[valid].corr(shifted_weather[valid]),
                            "n_pairs": int(valid.sum())})
    return pd.DataFrame(results)


def strongest_lags(correlations):
    return (correlations.assign(abs_correlation=correlations["correlation"].abs())
            .sort_values(["station_id", "variable", "abs_correlation"], ascending=[True, True, False])
            .drop_duplicates(["station_id", "variable"])
            .drop(columns="abs_correlation")
            .sort_values(["station_id", "correlation"], key=lambda s: s.abs() if s.name == "correlation" else s)
            .reset_index(drop=True))


pegel_data, weather_daily = load_data()
station_ids = sorted(weather_daily["station_id"].unique())
print(f"Loaded {len(pegel_data):,} hourly water observations and {len(weather_daily):,} daily station rows.")
print(f"Stations: {', '.join(station_ids)}")

Loaded 744,233 hourly water observations and 45,494 daily station rows.
Stations: 02074, 03278, 03402, 04931, 06275


In [9]:
# Subsample first: representative random rows exercise every station and lag.
SUBSAMPLE_ROWS = 10_000
subsample = pegel_data.sample(SUBSAMPLE_ROWS, random_state=42).sort_values("timestamp")
subsample_results = []
for station_id in station_ids:
    station_weather = weather_daily[weather_daily["station_id"] == station_id].drop(columns="station_id")
    result = correlation_by_lag(subsample, station_weather)
    result.insert(0, "station_id", station_id)
    subsample_results.append(result)
subsample_correlations = pd.concat(subsample_results, ignore_index=True)
display(subsample_correlations)
assert len(subsample_correlations) == len(station_ids) * 5 * (MAX_LAG_DAYS + 1)
assert subsample_correlations["n_pairs"].max() > 0

,station_id,variable,lag_days,correlation,n_pairs
0,02074,water_level,0,1.000000,10000
1,02074,humidity_relative_mean,0,0.241943,3170
2,02074,precipitation_amount,0,0.152847,3177
3,02074,pressure_vapor_mean,0,-0.229068,3170
4,02074,temperature_air_mean_2m,0,-0.285145,3176
...,...,...,...,...,...
95,06275,water_level,3,-0.024264,9928
96,06275,humidity_relative_mean,3,-0.101064,2612
97,06275,precipitation_amount,3,0.035657,2614
98,06275,pressure_vapor_mean,3,0.255867,2612


In [10]:
# Full-data correlations and visualizations, all saved in ../out.
all_results = []
for station_id in station_ids:
    station_weather = weather_daily[weather_daily["station_id"] == station_id].drop(columns="station_id")
    result = correlation_by_lag(pegel_data, station_weather)
    result.insert(0, "station_id", station_id)
    all_results.append(result)
correlations_by_station = pd.concat(all_results, ignore_index=True)
strongest_by_station = strongest_lags(correlations_by_station)
correlations_by_station.to_csv(OUT_DIR / "correlations_by_station_lag_0_to_3_days.csv", index=False)
strongest_by_station.to_csv(OUT_DIR / "strongest_correlations_by_station.csv", index=False)
subsample_correlations.to_csv(OUT_DIR / "subsample_correlations_by_station.csv", index=False)

variables = ["water_level", "humidity_relative_mean", "precipitation_amount",
             "pressure_vapor_mean", "temperature_air_mean_2m"]
fig, axes = plt.subplots(len(station_ids), 1, figsize=(10, 4 * len(station_ids)), squeeze=False)
for axis, station_id in zip(axes[:, 0], station_ids):
    matrix = correlations_by_station[correlations_by_station.station_id == station_id].pivot(
        index="variable", columns="lag_days", values="correlation"
    ).reindex(variables)
    image = axis.imshow(matrix, cmap="coolwarm", vmin=-1, vmax=1, aspect="auto")
    axis.set(title=f"Station {station_id}", xlabel="Lag (days)", ylabel="Variable",
             xticks=range(MAX_LAG_DAYS + 1), yticks=range(len(variables)), yticklabels=variables)
    for row in range(matrix.shape[0]):
        for col in range(matrix.shape[1]):
            axis.text(col, row, f"{matrix.iloc[row, col]:.2f}", ha="center", va="center")
#fig.colorbar(image, ax=axes[:, 0], label="Pearson correlation")
fig.tight_layout()
fig.savefig(OUT_DIR / "correlations_heatmaps_by_station.png", dpi=180, bbox_inches="tight")
plt.close(fig)

plot_data = strongest_by_station[strongest_by_station.variable != "water_level"]
fig, axis = plt.subplots(figsize=(11, 6))
for station_id, group in plot_data.groupby("station_id"):
    axis.plot(group["variable"], group["correlation"], marker="o", label=station_id)
axis.axhline(0, color="black", linewidth=0.8)
axis.set(title="Strongest absolute lagged correlation by weather station",
         xlabel="Variable", ylabel="Correlation", ylim=(-1, 1))
axis.tick_params(axis="x", rotation=25)
axis.legend(title="station_id")
fig.tight_layout()
fig.savefig(OUT_DIR / "strongest_correlations_by_station.png", dpi=180, bbox_inches="tight")
plt.close(fig)
display(strongest_by_station)


,station_id,variable,lag_days,correlation,n_pairs
0,02074,pressure_vapor_mean,0,-0.244149,232076
1,02074,humidity_relative_mean,1,0.251145,232076
2,02074,temperature_air_mean_2m,0,-0.289168,232388
3,02074,precipitation_amount,1,0.399712,232484
4,02074,water_level,0,1.000000,744233
5,03278,pressure_vapor_mean,0,-0.229282,199316
6,03278,humidity_relative_mean,1,0.230238,199316
7,03278,temperature_air_mean_2m,0,-0.280290,199676
8,03278,precipitation_amount,1,0.359785,199532
9,03278,water_level,0,1.000000,744233


In [11]:
# Hourly weather analysis: positive lag h compares water at t with weather at t-h hours.
MAX_LAG_HOURS = 24
weather_hourly = load_hourly_weather()
hourly_station_ids = sorted(weather_hourly["station_id"].unique())


def hourly_correlation_by_lag(pegel, weather_station, max_lag_hours=MAX_LAG_HOURS):
    weather_columns = [column for column in weather_station.columns if column not in {"station_id", "timestamp"}]
    base = pegel[["timestamp", "water_level"]].drop_duplicates("timestamp").sort_values("timestamp")
    weather_station = weather_station.drop_duplicates("timestamp").sort_values("timestamp")
    results = []
    for lag_hours in range(max_lag_hours + 1):
        lag = pd.Timedelta(hours=lag_hours)
        lagged_water = base.rename(columns={"water_level": "lagged_water"}).copy()
        lagged_water["timestamp"] += lag
        lagged_weather = weather_station.copy()
        lagged_weather["timestamp"] += lag
        joined = (base.merge(lagged_water, on="timestamp", how="left")
                       .merge(lagged_weather, on="timestamp", how="left"))
        valid_water = joined["water_level"].notna() & joined["lagged_water"].notna()
        results.append({"variable": "water_level", "lag_hours": lag_hours,
                        "correlation": joined.loc[valid_water, "water_level"].corr(joined.loc[valid_water, "lagged_water"]),
                        "n_pairs": int(valid_water.sum())})
        for column in weather_columns:
            valid = joined["water_level"].notna() & joined[column].notna()
            results.append({"variable": column, "lag_hours": lag_hours,
                            "correlation": joined.loc[valid, "water_level"].corr(joined.loc[valid, column]),
                            "n_pairs": int(valid.sum())})
    return pd.DataFrame(results)


hourly_subsample = pegel_data.sample(SUBSAMPLE_ROWS, random_state=42).sort_values("timestamp")
hourly_subsample_results = []
for station_id in hourly_station_ids:
    station_weather = weather_hourly[weather_hourly["station_id"] == station_id].drop(columns="station_id")
    result = hourly_correlation_by_lag(hourly_subsample, station_weather)
    result.insert(0, "station_id", station_id)
    hourly_subsample_results.append(result)
hourly_subsample_correlations = pd.concat(hourly_subsample_results, ignore_index=True)
display(hourly_subsample_correlations.head(20))
assert len(hourly_subsample_correlations) == len(hourly_station_ids) * 5 * (MAX_LAG_HOURS + 1)
assert hourly_subsample_correlations["n_pairs"].max() > 0

hourly_results = []
for station_id in hourly_station_ids:
    station_weather = weather_hourly[weather_hourly["station_id"] == station_id].drop(columns="station_id")
    result = hourly_correlation_by_lag(pegel_data, station_weather)
    result.insert(0, "station_id", station_id)
    hourly_results.append(result)
hourly_correlations_by_station = pd.concat(hourly_results, ignore_index=True)
hourly_strongest_by_station = (hourly_correlations_by_station.assign(abs_correlation=hourly_correlations_by_station["correlation"].abs())
    .sort_values(["station_id", "variable", "abs_correlation"], ascending=[True, True, False])
    .drop_duplicates(["station_id", "variable"])
    .drop(columns="abs_correlation")
    .reset_index(drop=True))
hourly_correlations_by_station.to_csv(OUT_DIR / "hourly_correlations_by_station_lag_0_to_24_hours.csv", index=False)
hourly_strongest_by_station.to_csv(OUT_DIR / "hourly_strongest_correlations_by_station.csv", index=False)
hourly_subsample_correlations.to_csv(OUT_DIR / "hourly_subsample_correlations_by_station.csv", index=False)

hourly_variables = ["water_level", "humidity_relative", "pressure_vapor", "temperature_air_2m", "precipitation_amount"]
fig, axes = plt.subplots(len(hourly_station_ids), 1, figsize=(11, 4 * len(hourly_station_ids)), squeeze=False)
for axis, station_id in zip(axes[:, 0], hourly_station_ids):
    matrix = hourly_correlations_by_station[hourly_correlations_by_station.station_id == station_id].pivot(
        index="variable", columns="lag_hours", values="correlation"
    ).reindex(hourly_variables)
    image = axis.imshow(matrix, cmap="coolwarm", vmin=-1, vmax=1, aspect="auto")
    axis.set(title=f"Hourly weather station {station_id}", xlabel="Lag (hours)", ylabel="Variable",
             xticks=range(0, MAX_LAG_HOURS + 1, 4), yticks=range(len(hourly_variables)), yticklabels=hourly_variables)
    axis.set_xticklabels(range(0, MAX_LAG_HOURS + 1, 4))
    for row in range(matrix.shape[0]):
        for col in range(0, matrix.shape[1], 4):
            axis.text(col, row, f"{matrix.iloc[row, col]:.2f}", ha="center", va="center", fontsize=8)
#fig.colorbar(image, ax=axes[:, 0], label="Pearson correlation")
fig.tight_layout()
fig.savefig(OUT_DIR / "hourly_correlations_heatmaps_by_station.png", dpi=180, bbox_inches="tight")
plt.close(fig)
display(hourly_strongest_by_station)


,station_id,variable,lag_hours,correlation,n_pairs
0,02074,water_level,0,1.000000,10000
1,02074,humidity_relative,0,0.162202,2659
2,02074,pressure_vapor,0,-0.212773,2659
3,02074,temperature_air_2m,0,-0.252836,2659
4,02074,precipitation_amount,0,0.123643,2665
5,02074,water_level,1,0.997224,143
6,02074,humidity_relative,1,0.150176,2657
7,02074,pressure_vapor,1,-0.217566,2657
8,02074,temperature_air_2m,1,-0.250488,2657
9,02074,precipitation_amount,1,0.067487,2665


,station_id,variable,lag_hours,correlation,n_pairs
0,02074,humidity_relative,14,0.183950,193015
1,02074,precipitation_amount,13,0.166776,193450
2,02074,pressure_vapor,0,-0.233948,193015
3,02074,temperature_air_2m,0,-0.261207,193015
4,02074,water_level,0,1.000000,744233
5,03278,humidity_relative,14,0.169891,198890
6,03278,precipitation_amount,13,0.145892,199204
7,03278,pressure_vapor,0,-0.222210,198890
8,03278,temperature_air_2m,0,-0.256638,198890
9,03278,water_level,0,1.000000,744233
